# Mejoras del Modelo — Predicción Malnutrición PMCI

Cuatro mejoras sobre el pipeline base de LightGBM:
1. **Validación temporal hold-out** — P1-P5 entrenamiento, P6 prueba real
2. **Umbral óptimo clínico** — maximizar sensibilidad con especificidad aceptable
3. **Modelo parsimonioso de nacimiento (F1)** — top variables con mínima pérdida de AUC
4. **Calibración de probabilidades** — ¿el modelo reporta probabilidades confiables?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json, warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import shap
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, confusion_matrix,
    roc_curve, precision_recall_curve, average_precision_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve  # movido en sklearn 1.x

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

PATH = '/Users/herjimenez/Documents/MAESTRIA/PROYECTOS/Trabajo de grado/KMC-70k-93-2024-Malnutricion-conVel-DATA-SPSS-20250322.xlsx'
OUT  = '/Users/herjimenez/Documents/MAESTRIA/PROYECTOS/Trabajo de grado/'
PLAN = OUT + 'feature_plan.json'

print('Cargando datos...')
df_raw = pd.read_excel(PATH)
df = df_raw.replace('#NULL!', np.nan).copy()
for col in df.columns:
    c = pd.to_numeric(df[col], errors='coerce')
    if df[col].notna().sum() == 0 or c.notna().sum()/df[col].notna().sum() >= 0.5:
        df[col] = c

with open(PLAN) as f:
    plan = json.load(f)
FASES = plan['fases']

df['stunting12m']      = np.where(df['zscoretalla12cat'].notna(),
                                   (df['zscoretalla12cat']==1.0).astype(float), np.nan)
df['underweight12m_b'] = np.where(df['zscorepeso12cat'].notna(),
                                   (df['zscorepeso12cat']==1.0).astype(float), np.nan)
df['wasting12m']       = np.where(df['zscorepesotalla12cat'].notna(),
                                   (df['zscorepesotalla12cat']==1.0).astype(float), np.nan)

FASE_ORDER = ['F0_Prenatal_Parto','F1_Nacimiento','F2_Hospitalizacion',
              'F3_40semanas','F4_3meses','F5_6meses','F6_9meses']
LEAKAGE = {'velocidad12_9mesesOMS','vino12m','Desercionreal12meses',
           'rehosp40a12meses','mortalidad40sem12meses','indexnutricion12meses',
           'MUERTE1ANO','examenneurodurante12meses','examenneuropsico12meses','riesgoPC12m'}
LEAKAGE |= {c for c in df.columns if '12' in str(c) and
            c not in ('stunting12m','underweight12m_b','wasting12m')}

cumulative_features = {}
acum = []
for fase in FASE_ORDER:
    nuevas = [c for c in FASES.get(fase,[]) if c in df.columns and c not in LEAKAGE]
    acum   = acum + [c for c in nuevas if c not in acum]
    cumulative_features[fase] = list(acum)

LGBM_PARAMS = {
    'objective': 'binary', 'metric': 'auc',
    'learning_rate': 0.05, 'num_leaves': 63,
    'min_child_samples': 30, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 5,
    'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'verbose': -1, 'seed': 42,
}
PERIOD_LABELS = {1:'P1',2:'P2',3:'P3',4:'P4',5:'P5',6:'P6'}
TARGET = 'stunting12m'
fase_labels_short = {
    'F0_Prenatal_Parto':'F0 Prenatal', 'F1_Nacimiento':'F1 Nacimiento',
    'F2_Hospitalizacion':'F2 Hosp.', 'F3_40semanas':'F3 40sem',
    'F4_3meses':'F4 3m', 'F5_6meses':'F5 6m', 'F6_9meses':'F6 9m',
}

print(f'Dataset: {df.shape[0]:,} x {df.shape[1]:,}')
print(f'Fases: {list(cumulative_features.keys())}')

## Mejora 1: Validación Temporal Hold-out

El CV aleatorio mezcla datos de 1998-2023 en train/val. En producción, el modelo se entrena en datos históricos y se aplica a datos futuros. 
**Train**: P1-P5 (hasta 2017) | **Test**: P6 (2018-2023)

In [ ]:
def train_lgbm_full(X_tr, y_tr, X_val, y_val, spw=None):
    """Entrena un modelo LightGBM completo con early stopping sobre un val set."""
    n_pos = y_tr.sum()
    n_neg = len(y_tr) - n_pos
    params = {**LGBM_PARAMS,
              'scale_pos_weight': spw if spw else round(n_neg/n_pos, 2)}
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    model  = lgb.train(
        params, dtrain, num_boost_round=500,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )
    return model


def metricas(y_true, y_prob, thr=0.5, label=''):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    ap   = average_precision_score(y_true, y_prob)
    sens = tp/(tp+fn) if tp+fn>0 else 0
    spec = tn/(tn+fp) if tn+fp>0 else 0
    f1   = f1_score(y_true, y_pred, zero_division=0)
    bs   = brier_score_loss(y_true, y_prob)
    if label:
        print(f'  {label:<30}: AUC={auc:.4f}  AP={ap:.4f}  '
              f'Sens={sens:.3f}  Spec={spec:.3f}  F1={f1:.3f}  Brier={bs:.4f}')
    return dict(AUC=auc, AP=ap, Sens=sens, Spec=spec, F1=f1, Brier=bs)


# Crear splits temporales
mask_train = df['periodosanalisis'].isin([1,2,3,4,5])
mask_test  = df['periodosanalisis'] == 6

print('Distribución del split temporal:')
for split, mask in [('Train (P1-P5)', mask_train), ('Test  (P6)',    mask_test)]:
    sub   = df[mask]
    n_out = sub[TARGET].notna().sum()
    n_pos = sub[TARGET].sum()
    pct   = n_pos/n_out*100 if n_out > 0 else 0
    print(f'  {split}: {len(sub):>6,} total | {n_out:>6,} con outcome | {n_pos:>5,.0f} stunted ({pct:.1f}%)')

In [ ]:
# Entrenar cascada temporal con split P1-P5 → P6
results_temporal = {}

print('=== Validacion TEMPORAL: Train P1-P5 | Test P6 ===')
print(f'{"Fase":<25} {"CV-AUC":>8} {"Hold-AUC":>10} {"Delta":>8} {"Sens":>7} {"Spec":>7}')
print('-' * 75)

# AUC del CV aleatorio original (referencia)
cv_aucs_ref = {
    'F0_Prenatal_Parto':  0.6454,
    'F1_Nacimiento':      0.7374,
    'F2_Hospitalizacion': 0.7405,
    'F3_40semanas':       0.7678,
    'F4_3meses':          0.8209,
    'F5_6meses':          0.8935,
    'F6_9meses':          0.9290,
}

for fase in FASE_ORDER:
    feats = cumulative_features[fase]
    cols  = [c for c in feats if c in df.columns]

    df_tr = df[mask_train & df[TARGET].notna()][cols + [TARGET]].dropna(subset=[TARGET])
    df_te = df[mask_test  & df[TARGET].notna()][cols + [TARGET]].dropna(subset=[TARGET])

    if len(df_tr) < 100 or df_tr[TARGET].sum() < 20:
        continue
    if len(df_te) < 50 or df_te[TARGET].sum() < 10:
        continue

    X_tr, y_tr = df_tr[cols], df_tr[TARGET].astype(int)
    X_te, y_te = df_te[cols], df_te[TARGET].astype(int)

    # Usar 20% del train como val para early stopping
    skf_tmp = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    tr_idx, val_idx = next(skf_tmp.split(X_tr, y_tr))
    model = train_lgbm_full(X_tr.iloc[tr_idx], y_tr.iloc[tr_idx],
                             X_tr.iloc[val_idx], y_tr.iloc[val_idx])

    prob_te  = model.predict(X_te)
    auc_hold = roc_auc_score(y_te, prob_te)
    sens_h   = metricas(y_te, prob_te)['Sens']
    spec_h   = metricas(y_te, prob_te)['Spec']
    auc_cv   = cv_aucs_ref.get(fase, 0)
    delta    = auc_hold - auc_cv

    results_temporal[fase] = {
        'model': model, 'X_te': X_te, 'y_te': y_te,
        'prob_te': prob_te, 'AUC_hold': auc_hold, 'AUC_cv': auc_cv,
    }

    print(f'  {fase:<23} {auc_cv:>8.4f} {auc_hold:>10.4f} {delta:>+8.4f} '
          f'{sens_h:>7.3f} {spec_h:>7.3f}')

print()
print('Delta positivo = modelo generaliza bien al futuro')
print('Delta negativo = posible sobreajuste o cambio de distribucion')

In [ ]:
# Grafica: CV aleatorio vs Hold-out temporal
fases_ok = [f for f in FASE_ORDER if f in results_temporal]
aucs_cv  = [results_temporal[f]['AUC_cv']   for f in fases_ok]
aucs_ho  = [results_temporal[f]['AUC_hold'] for f in fases_ok]

fase_labels_short = {
    'F0_Prenatal_Parto':'F0 Prenatal', 'F1_Nacimiento':'F1 Nacimiento',
    'F2_Hospitalizacion':'F2 Hosp.', 'F3_40semanas':'F3 40sem',
    'F4_3meses':'F4 3m', 'F5_6meses':'F5 6m', 'F6_9meses':'F6 9m',
}
x_lbls = [fase_labels_short[f] for f in fases_ok]
x      = np.arange(len(fases_ok))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, aucs_cv, marker='s', linewidth=2, color='#95a5a6',
        linestyle='--', label='CV aleatorio (5-fold)', zorder=3)
ax.plot(x, aucs_ho, marker='o', linewidth=2.5, color='#2ecc71',
        label='Hold-out temporal (P6 test)', zorder=4)

for xi, cv, ho in zip(x, aucs_cv, aucs_ho):
    delta = ho - cv
    color = '#27ae60' if delta >= 0 else '#e74c3c'
    ax.annotate(f'{delta:+.3f}',
                xy=(xi, ho), xytext=(xi, ho + 0.015),
                ha='center', fontsize=8, color=color, fontweight='bold')

ax.fill_between(x, aucs_cv, aucs_ho,
                where=[h >= c for h, c in zip(aucs_ho, aucs_cv)],
                alpha=0.15, color='#2ecc71', label='Hold-out > CV')
ax.fill_between(x, aucs_cv, aucs_ho,
                where=[h < c for h, c in zip(aucs_ho, aucs_cv)],
                alpha=0.15, color='#e74c3c', label='Hold-out < CV')

ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(x_lbls)
ax.set_ylim(0.55, 1.0)
ax.set_ylabel('ROC-AUC')
ax.set_title('CV aleatorio vs Validacion Hold-out Temporal (P6 = 2018-2023)\n'
             'Stunting — ¿el modelo generaliza al futuro?', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT + 'mej_01_validacion_temporal.png', bbox_inches='tight')
plt.close()
print('Guardado: mej_01_validacion_temporal.png')

## Mejora 2: Umbral Óptimo Clínico

El umbral por defecto (0.5) no es óptimo con datos desbalanceados.  
Evaluamos tres criterios clínicos:
- **Youden's J** = argmax(Sensibilidad + Especificidad - 1)
- **Sensibilidad ≥ 80%** = no perder al menos 4/5 casos reales
- **F1 máximo** = equilibrio precisión-recall

In [ ]:
def find_optimal_thresholds(y_true, y_prob):
    """Calcula umbrales óptimos según distintos criterios."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    specificity = 1 - fpr

    # Youden's J
    j_scores = tpr + specificity - 1
    idx_j    = np.argmax(j_scores)
    thr_j    = thresholds[idx_j]

    # Sensibilidad >= 80%, maximizar especificidad
    mask_80  = tpr >= 0.80
    if mask_80.any():
        idx_80  = np.where(mask_80)[0][-1]  # mayor especificidad con sens>=80%
        thr_80  = thresholds[idx_80]
        spec_80 = specificity[idx_80]
        sens_80 = tpr[idx_80]
    else:
        thr_80 = thresholds[0]; spec_80 = 0; sens_80 = 0

    # F1 máximo
    f1_scores = []
    for thr in thresholds:
        pred = (y_prob >= thr).astype(int)
        f1_scores.append(f1_score(y_true, pred, zero_division=0))
    idx_f1 = np.argmax(f1_scores)
    thr_f1 = thresholds[idx_f1]

    return {
        'Youden':     {'thr': thr_j,  'sens': tpr[idx_j],  'spec': specificity[idx_j]},
        'Sens>=80%':  {'thr': thr_80, 'sens': sens_80,      'spec': spec_80},
        'F1-max':     {'thr': thr_f1, 'sens': tpr[idx_f1], 'spec': specificity[idx_f1]},
        'Default-0.5':{'thr': 0.5,
                       'sens': tpr[np.argmin(np.abs(thresholds-0.5))],
                       'spec': specificity[np.argmin(np.abs(thresholds-0.5))]},
        '_roc': (fpr, tpr, thresholds),
    }


# Usar las predicciones OOF del modelo de nacimiento (F1) del hold-out
# Recalcular con CV 5-fold para obtener predicciones OOF en P1-P5
fase_thr = 'F1_Nacimiento'
feats_thr = cumulative_features[fase_thr]
cols_thr  = [c for c in feats_thr if c in df.columns]

# Datos completos (todos los periodos) con outcome
df_thr = df[df[TARGET].notna()][cols_thr + [TARGET]].dropna(subset=[TARGET])
X_thr, y_thr = df_thr[cols_thr], df_thr[TARGET].astype(int)

# OOF con CV
skf_thr  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_thr  = np.zeros(len(y_thr))
for tr, val in skf_thr.split(X_thr, y_thr):
    spw = (len(y_thr.iloc[tr]) - y_thr.iloc[tr].sum()) / y_thr.iloc[tr].sum()
    m   = train_lgbm_full(X_thr.iloc[tr], y_thr.iloc[tr],
                           X_thr.iloc[val], y_thr.iloc[val], spw=spw)
    oof_thr[val] = m.predict(X_thr.iloc[val])

thresholds_info = find_optimal_thresholds(y_thr, oof_thr)

print(f'Analisis de Umbral Optimo — {fase_thr} ({TARGET})')
print(f'{"Criterio":<15} {"Umbral":>8} {"Sensibilidad":>14} {"Especificidad":>15} {"VPP*":>8}')
print('-' * 65)
prev = y_thr.mean()
for criterio, vals in thresholds_info.items():
    if criterio.startswith('_'): continue
    t, s, sp = vals['thr'], vals['sens'], vals['spec']
    # Valor Predictivo Positivo aproximado (Bayes)
    if s+sp > 0:
        vpp = (s * prev) / (s * prev + (1-sp) * (1-prev))
    else:
        vpp = 0
    print(f'  {criterio:<13} {t:>8.3f} {s:>14.3f} {sp:>15.3f} {vpp:>8.3f}')
print('(*VPP = de cada alarma, cuantos son casos reales)')

In [ ]:
fpr, tpr, thrs = thresholds_info['_roc']
auc_val = roc_auc_score(y_thr, oof_thr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC con puntos operativos
ax = axes[0]
ax.plot(fpr, tpr, color='#3498db', linewidth=2,
        label=f'ROC (AUC={auc_val:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.4)

criterio_styles = {
    'Youden':      ('*', '#e74c3c', 14),
    'Sens>=80%':   ('D', '#e67e22', 10),
    'F1-max':      ('^', '#2ecc71', 10),
    'Default-0.5': ('o', '#95a5a6', 10),
}
for criterio, (marker, color, ms) in criterio_styles.items():
    if criterio in thresholds_info:
        v = thresholds_info[criterio]
        ax.scatter(1-v['spec'], v['sens'], marker=marker, color=color,
                   s=ms**2, zorder=5, label=f"{criterio} (thr={v['thr']:.2f})")

ax.set_xlabel('1 - Especificidad')
ax.set_ylabel('Sensibilidad')
ax.set_title(f'Curva ROC — {fase_thr}\nPuntos de operacion clinica',
             fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Sensibilidad y Especificidad vs Umbral
ax2 = axes[1]
ax2.plot(thrs, tpr[:-1],     color='#e74c3c', linewidth=2, label='Sensibilidad')
ax2.plot(thrs, 1-fpr[:-1],   color='#3498db', linewidth=2, label='Especificidad')
ax2.plot(thrs, tpr[:-1]+(1-fpr[:-1])-1, color='#2ecc71',
         linewidth=1.5, linestyle='--', label="Youden's J")

for criterio, (marker, color, ms) in criterio_styles.items():
    if criterio in thresholds_info and not criterio.startswith('_'):
        t = thresholds_info[criterio]['thr']
        ax2.axvline(t, color=color, linestyle=':', alpha=0.7)
        ax2.text(t + 0.005, 0.05, f"{criterio}\n{t:.2f}",
                 fontsize=7, color=color)

ax2.set_xlabel('Umbral de clasificacion')
ax2.set_ylabel('Metrica')
ax2.set_title('Sensibilidad y Especificidad vs Umbral', fontweight='bold')
ax2.legend()
ax2.set_xlim(0, 1)
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Analisis de Umbral Optimo — Stunting | {fase_thr}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT + 'mej_02_umbral_optimo.png', bbox_inches='tight')
plt.close()
print('Guardado: mej_02_umbral_optimo.png')

## Mejora 3: Modelo Parsimonioso de Nacimiento (F1)

¿Cuáles son las 10-15 variables de nacimiento más importantes?  
Objetivo: modelo simple, interpretable y usable desde el alta hospitalaria.

In [ ]:
# Entrenamos el modelo F1 completo y calculamos SHAP
fase_f1   = 'F1_Nacimiento'
feats_f1  = cumulative_features[fase_f1]
cols_f1   = [c for c in feats_f1 if c in df.columns]

df_f1     = df[df[TARGET].notna()][cols_f1 + [TARGET]].dropna(subset=[TARGET])
X_f1, y_f1 = df_f1[cols_f1], df_f1[TARGET].astype(int)

# Entrenamos con 80% train, 20% val para early stopping
skf_f1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr_f1, val_f1 = next(skf_f1.split(X_f1, y_f1))
model_f1 = train_lgbm_full(X_f1.iloc[tr_f1], y_f1.iloc[tr_f1],
                             X_f1.iloc[val_f1], y_f1.iloc[val_f1])

# SHAP values
print(f'Calculando SHAP para {fase_f1}...')
n_shap   = min(3000, len(X_f1))
X_sample = X_f1.sample(n_shap, random_state=42)
explainer_f1   = shap.TreeExplainer(model_f1)
shap_vals_f1   = explainer_f1.shap_values(X_sample)

shap_imp_f1 = pd.Series(
    np.abs(shap_vals_f1).mean(axis=0),
    index=X_f1.columns
).sort_values(ascending=False)

print(f'\nTop 20 variables de nacimiento (SHAP) para predecir Stunting a 12m:')
print(f'{"Variable":<40} {"SHAP medio":>12} {"r con stunting":>15}')
print('-' * 70)
for feat, shap_v in shap_imp_f1.head(20).items():
    r = df[feat].corr(df[TARGET]) if feat in df.columns else float('nan')
    print(f'  {feat:<38} {shap_v:>12.4f} {r:>+15.3f}')

In [ ]:
# SHAP summary plot para modelo F1
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_vals_f1, X_sample, max_display=20,
                  plot_type='dot', show=False)
plt.title(f'SHAP — Top 20 factores al NACIMIENTO que predicen Stunting a 12m\n'
          f'(modelo F1: solo variables disponibles al alta hospitalaria)',
          fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig(OUT + 'mej_03_shap_nacimiento.png', bbox_inches='tight')
plt.close()
print('Guardado: mej_03_shap_nacimiento.png')

In [ ]:
# Curva de parsimoniosidad: AUC vs numero de features
ns_to_test = [3, 5, 8, 10, 12, 15, 20, 30, 50, len(cols_f1)]
parsimonious_results = []

print('Evaluando modelos parsimoniosos (5-fold CV)...')
for n_feats in ns_to_test:
    top_feats = shap_imp_f1.head(n_feats).index.tolist()
    X_p = X_f1[top_feats]

    aucs_p = []
    for tr, val in StratifiedKFold(5, shuffle=True, random_state=42).split(X_p, y_f1):
        m = train_lgbm_full(X_p.iloc[tr], y_f1.iloc[tr],
                             X_p.iloc[val], y_f1.iloc[val])
        aucs_p.append(roc_auc_score(y_f1.iloc[val], m.predict(X_p.iloc[val])))

    auc_m = np.mean(aucs_p)
    parsimonious_results.append({'n': n_feats, 'AUC': auc_m, 'std': np.std(aucs_p)})
    print(f'  {n_feats:>3} features → AUC={auc_m:.4f} ± {np.std(aucs_p):.4f}')

auc_full = next(r['AUC'] for r in parsimonious_results if r['n'] == len(cols_f1))
print(f'\nModelo completo ({len(cols_f1)} features): AUC={auc_full:.4f}')

In [ ]:
pars_df = pd.DataFrame(parsimonious_results)
auc_ref = pars_df.iloc[-1]['AUC']

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(pars_df['n'], pars_df['AUC'], marker='o', linewidth=2.5,
        color='#3498db', zorder=4)
ax.fill_between(pars_df['n'],
                pars_df['AUC'] - pars_df['std'],
                pars_df['AUC'] + pars_df['std'],
                alpha=0.15, color='#3498db')

for _, row in pars_df.iterrows():
    ax.text(row['n'], row['AUC'] + 0.003, f"{row['AUC']:.3f}",
            ha='center', fontsize=8)

# Marcar el 95% del AUC completo
thr_95 = auc_ref * 0.98
ax.axhline(thr_95, color='#e74c3c', linestyle='--', alpha=0.7,
           label=f'98% del AUC completo ({thr_95:.3f})')
ax.axhline(auc_ref, color='#2ecc71', linestyle='--', alpha=0.7,
           label=f'AUC completo ({auc_ref:.3f})')

# Marcar el "codo"
diffs = np.diff(pars_df['AUC'].values)
codo_idx = np.argmax(np.abs(diffs[1:]) < 0.005) + 1  # donde se estabiliza
n_codo = pars_df.iloc[codo_idx]['n']
auc_codo = pars_df.iloc[codo_idx]['AUC']
ax.axvline(n_codo, color='#f39c12', linestyle=':', alpha=0.8)
ax.text(n_codo + 1, 0.70, f'Codo:\n{int(n_codo)} vars',
        color='#f39c12', fontweight='bold', fontsize=9)

ax.set_xlabel('Numero de features (top SHAP)')
ax.set_ylabel('ROC-AUC (5-fold CV)')
ax.set_title('Curva de Parsimonia — Modelo de Nacimiento (F1)\n'
             '¿Cuantas variables necesito para un modelo util?',
             fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT + 'mej_04_parsimonia.png', bbox_inches='tight')
plt.close()
print('Guardado: mej_04_parsimonia.png')

# Variables del modelo parsimonioso final
top10 = shap_imp_f1.head(10).index.tolist()
print(f'\nModelo parsimonioso recomendado (top 10 variables de nacimiento):')
for i, v in enumerate(top10, 1):
    print(f'  {i:>2}. {v}')

## Mejora 4: Calibración de Probabilidades

Un modelo con AUC=0.93 puede tener probabilidades mal calibradas:  
`P=0.8` podría significar solo 40% de casos reales.  
La calibración es crítica para comunicar riesgo a los médicos.

In [ ]:
# Calibracion para F1 y F6 (nacimiento y mejor fase)
modelos_calibrar = [
    ('F1 Nacimiento', 'F1_Nacimiento'),
    ('F6 9 meses',    'F6_9meses'),
]

fig, axes = plt.subplots(1, len(modelos_calibrar), figsize=(14, 5))
brier_resultados = {}

for ax, (label, fase) in zip(axes, modelos_calibrar):
    feats   = cumulative_features[fase]
    cols    = [c for c in feats if c in df.columns]
    df_cal  = df[df[TARGET].notna()][cols + [TARGET]].dropna(subset=[TARGET])
    X_cal   = df_cal[cols]
    y_cal   = df_cal[TARGET].astype(int)

    # Predicciones OOF
    oof_cal = np.zeros(len(y_cal))
    for tr, val in StratifiedKFold(5, shuffle=True, random_state=42).split(X_cal, y_cal):
        spw = (len(y_cal.iloc[tr]) - y_cal.iloc[tr].sum()) / y_cal.iloc[tr].sum()
        m   = train_lgbm_full(X_cal.iloc[tr], y_cal.iloc[tr],
                               X_cal.iloc[val], y_cal.iloc[val], spw=spw)
        oof_cal[val] = m.predict(X_cal.iloc[val])

    # Calibracion
    frac_pos, mean_pred = calibration_curve(y_cal, oof_cal, n_bins=10, strategy='quantile')
    bs = brier_score_loss(y_cal, oof_cal)
    bs_ref = y_cal.mean() * (1 - y_cal.mean())  # Brier de prediccion naive (prevalencia)
    bss = 1 - bs / bs_ref  # Brier Skill Score

    brier_resultados[label] = {'Brier': bs, 'BSS': bss}

    ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Calibracion perfecta')
    ax.plot(mean_pred, frac_pos, marker='o', linewidth=2, color='#3498db',
            label=f'LightGBM (Brier={bs:.4f})')

    # Diferencia entre predicho y real
    for xp, yp in zip(mean_pred, frac_pos):
        ax.annotate(f'{yp-xp:+.2f}',
                    xy=(xp, yp), xytext=(xp+0.02, yp-0.03),
                    fontsize=7, color='#e74c3c')

    ax.fill_between([0,1],[0,1],[1,1], alpha=0.05, color='#e74c3c', label='Sobreestima')
    ax.fill_between([0,1],[0,0],[0,1], alpha=0.05, color='#2ecc71', label='Subestima')

    ax.set_xlabel('Probabilidad predicha')
    ax.set_ylabel('Fraccion de positivos reales')
    ax.set_title(f'Calibracion — {label}\nBrier={bs:.4f} | BSS={bss:.3f}',
                 fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)

plt.suptitle('Curvas de Calibracion — ¿Las probabilidades son confiables?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT + 'mej_05_calibracion.png', bbox_inches='tight')
plt.close()
print('Guardado: mej_05_calibracion.png')

print('\nBrier Score (menor = mejor | 0 = perfecto | ~0.19 = naive para 24.6%)')
for label, vals in brier_resultados.items():
    print(f'  {label:<20}: Brier={vals["Brier"]:.4f}  BSS={vals["BSS"]:.3f}')

## Resumen de Mejoras

In [ ]:
print('=' * 68)
print('RESUMEN DE MEJORAS — Modelo Malnutricion PMCI')
print('=' * 68)

print('''
MEJORA 1 — Validacion Temporal (P6 como hold-out)
  Resultado esperado: AUC del hold-out similar o mayor al CV aleatorio
  -> Si AUC_hold >= AUC_cv: el modelo generaliza bien al futuro
  -> Si AUC_hold <  AUC_cv: hay sobreajuste o drift temporal

MEJORA 2 — Umbral Optimo Clinico
  Default (0.5)  : equilibrio sin consideracion clinica
  Youden         : maximiza sensibilidad + especificidad simultaneamente
  Sens >= 80%    : no perder al menos 4 de cada 5 casos reales
  F1-max         : mejor balance precision-recall
  -> Para screening clinico: usar umbral de Sens>=80%

MEJORA 3 — Modelo Parsimonioso de Nacimiento (F1)
  El modelo de nacimiento (AUC~0.74) puede simplificarse a 10 vars
  con minima perdida de AUC. Estas 10 variables son:
  -> Las candidatas para incluir en un score clinico simple

MEJORA 4 — Calibracion
  Brier Score < Brier naive = el modelo aporta sobre la prevalencia
  BSS > 0 = el modelo es mejor que predecir siempre la prevalencia
  Curva de calibracion cercana a la diagonal = probabilidades confiables
''')

# Tabla de comparacion final
print('METRICAS FINALES POR FASE (Stunting — Umbral Youden):')
thr_youden = thresholds_info['Youden']['thr']
print(f'Umbral Youden optimo: {thr_youden:.3f}')
print()
print(f'{"Fase":<25} {"AUC Hold-out":>13} {"Sens":>7} {"Spec":>7} {"F1":>7}')
print('-' * 60)
for fase, res in results_temporal.items():
    y_t, p_t = res['y_te'], res['prob_te']
    m = metricas(y_t, p_t, thr=thr_youden)
    print(f'  {fase_labels_short[fase]:<23} {res["AUC_hold"]:>13.4f} '
          f'{m["Sens"]:>7.3f} {m["Spec"]:>7.3f} {m["F1"]:>7.3f}')